# 🚗 ParkVision – 02: Model Training
**ITAI 1378 – Computer Vision & AI**

This notebook trains YOLOv8 on the PKLot dataset to detect occupied vs. empty parking spaces.

### Notebook Flow
1. ✅ Environment check
2. 📦 Reinstall dependencies & reload dataset
3. 🔍 Verify dataset structure
4. 🏋️ Train YOLOv8
5. 📊 Evaluate results
6. 🖼️ Run inference on test images
7. 💾 Save model to Google Drive

---
## 1. Environment Check
Always run this first. Make sure GPU shows as available before proceeding.

In [ ]:
import torch

print('=== GPU CHECK ===')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print('✅ Good to go!')
else:
    print('❌ No GPU detected!')
    print('Go to Runtime → Change Runtime Type → T4 GPU → Save')
    print('Then restart and rerun this cell.')

---
## 2. Install Dependencies & Load Dataset
Colab resets every session — we need to reinstall and re-download each time.
Your actual data is safe in Google Drive from the previous notebook.

In [ ]:
!pip install ultralytics roboflow -q
print('✅ Packages installed')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
import os

# ─── OPTION A: Load from Google Drive (fastest — no re-download) ───
DATASET_PATH = '/content/drive/MyDrive/ParkVision/dataset'

if os.path.exists(DATASET_PATH):
    print(f'✅ Dataset found in Drive: {DATASET_PATH}')
else:
    # ─── OPTION B: Re-download from Roboflow ───
    print('Dataset not found in Drive — downloading from Roboflow...')
    from roboflow import Roboflow

    # PASTE YOUR ROBOFLOW API KEY BELOW
    rf = Roboflow(api_key="YOUR_API_KEY_HERE")
    project = rf.workspace("sagitova-aliya").project("pklot-qesrf")
    dataset = project.version(1).download("yolov8")
    DATASET_PATH = dataset.location
    print(f'✅ Downloaded to: {DATASET_PATH}')

print(f'\nDataset path: {DATASET_PATH}')

---
## 3. Verify Dataset Structure
Before training, confirm everything looks correct.

In [ ]:
import glob

print('=== DATASET STRUCTURE ===')
for split in ['train', 'valid', 'test']:
    img_dir = os.path.join(DATASET_PATH, split, 'images')
    lbl_dir = os.path.join(DATASET_PATH, split, 'labels')
    if os.path.exists(img_dir):
        n_imgs = len(glob.glob(img_dir + '/*.jpg') + glob.glob(img_dir + '/*.png'))
        n_lbls = len(glob.glob(lbl_dir + '/*.txt')) if os.path.exists(lbl_dir) else 0
        print(f'  {split:8s}: {n_imgs:5d} images | {n_lbls:5d} labels')
    else:
        print(f'  {split:8s}: ❌ NOT FOUND')

# Check data.yaml exists
yaml_path = os.path.join(DATASET_PATH, 'data.yaml')
if os.path.exists(yaml_path):
    print(f'\n✅ data.yaml found')
    with open(yaml_path, 'r') as f:
        print(f.read())
else:
    print('\n❌ data.yaml NOT found — check your dataset path!')

In [ ]:
import cv2
import matplotlib.pyplot as plt
import random

# Preview 6 random training images with their labels
img_files = glob.glob(os.path.join(DATASET_PATH, 'train/images/*.jpg'))
img_files += glob.glob(os.path.join(DATASET_PATH, 'train/images/*.png'))
sample = random.sample(img_files, min(6, len(img_files)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, img_path in zip(axes.flatten(), sample):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Draw bounding boxes from label file
    lbl_path = img_path.replace('images', 'labels').replace('.jpg', '.txt').replace('.png', '.txt')
    h, w = img.shape[:2]
    class_colors = {0: (255, 80, 80), 1: (80, 255, 80)}  # 0=occupied red, 1=empty green
    class_names = {0: 'occupied', 1: 'empty'}

    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, cx, cy, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = int((cx - bw/2) * w)
                    y1 = int((cy - bh/2) * h)
                    x2 = int((cx + bw/2) * w)
                    y2 = int((cy + bh/2) * h)
                    color = class_colors.get(cls, (255, 255, 0))
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)

    ax.imshow(img)
    ax.set_title(os.path.basename(img_path), fontsize=8)
    ax.axis('off')

plt.suptitle('Sample Training Images (Red=Occupied, Green=Empty)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Train YOLOv8

We're using **YOLOv8s** (small) — good balance of speed and accuracy for Tier 2.

| Model | Speed | Accuracy | Use when... |
|-------|-------|----------|-------------|
| yolov8n | ⚡⚡⚡ | ⭐⭐ | Colab keeps crashing |
| yolov8s | ⚡⚡ | ⭐⭐⭐ | **← We use this** |
| yolov8m | ⚡ | ⭐⭐⭐⭐ | If accuracy is too low |

Training takes ~15–30 mins on a T4 GPU.

In [ ]:
# ─── TRAINING CONFIGURATION ───
# Adjust these if needed

CONFIG = {
    'model':      'yolov8s.pt',      # pretrained YOLOv8 small
    'data':       yaml_path,          # path to data.yaml
    'epochs':     50,                 # number of training rounds
    'imgsz':      640,                # image size (px)
    'batch':      16,                 # images per batch (reduce to 8 if OOM error)
    'patience':   10,                 # early stopping patience
    'project':    'ParkVision',       # output folder name
    'name':       'train_v1',         # run name
    'exist_ok':   True,
    'pretrained': True,               # use COCO pretrained weights
    'optimizer':  'AdamW',
    'lr0':        0.001,              # initial learning rate
    'augment':    True,               # enable built-in augmentation
    'cache':      False,              # set True if you have lots of RAM
    'device':     0,                  # GPU device (0 = first GPU)
    'workers':    2,
    'verbose':    True,
}

print('=== TRAINING CONFIGURATION ===')
for k, v in CONFIG.items():
    print(f'  {k:12s}: {v}')

In [ ]:
from ultralytics import YOLO

# Load the pretrained model
model = YOLO(CONFIG['model'])

print('🚀 Starting training...')
print('This will take 15–30 minutes on a T4 GPU. Grab a coffee!')
print('Watch the mAP50 column — we want it to reach 0.85+')
print('-' * 60)

results = model.train(
    data      = CONFIG['data'],
    epochs    = CONFIG['epochs'],
    imgsz     = CONFIG['imgsz'],
    batch     = CONFIG['batch'],
    patience  = CONFIG['patience'],
    project   = CONFIG['project'],
    name      = CONFIG['name'],
    exist_ok  = CONFIG['exist_ok'],
    pretrained= CONFIG['pretrained'],
    optimizer = CONFIG['optimizer'],
    lr0       = CONFIG['lr0'],
    augment   = CONFIG['augment'],
    cache     = CONFIG['cache'],
    device    = CONFIG['device'],
    workers   = CONFIG['workers'],
    verbose   = CONFIG['verbose'],
)

print('\n✅ Training complete!')

# Path to best weights
BEST_MODEL_PATH = f"ParkVision/{CONFIG['name']}/weights/best.pt"
print(f'Best model saved at: {BEST_MODEL_PATH}')

---
## 5. Evaluate Results
Check how well your model performs on the validation set.

In [ ]:
# Load best model and run validation
best_model = YOLO(BEST_MODEL_PATH)

print('📊 Running validation...')
metrics = best_model.val(data=CONFIG['data'], imgsz=CONFIG['imgsz'], device=0)

print('\n=== VALIDATION RESULTS ===')
print(f"mAP50:        {metrics.box.map50:.4f}   (target: ≥ 0.85)")
print(f"mAP50-95:     {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

# Simple pass/fail
map50 = metrics.box.map50
print('\n=== GRADE CHECK ===')
if map50 >= 0.90:
    print(f'🏆 EXCELLENT! mAP50={map50:.3f} — well above target')
elif map50 >= 0.85:
    print(f'✅ GOOD! mAP50={map50:.3f} — meets target')
elif map50 >= 0.70:
    print(f'⚠️  OKAY. mAP50={map50:.3f} — below target, try tuning')
else:
    print(f'❌ LOW. mAP50={map50:.3f} — see troubleshooting section below')

In [ ]:
# Plot training curves — look for smooth decline in loss, rise in mAP
from IPython.display import Image as IPImage, display

results_dir = f"ParkVision/{CONFIG['name']}"
results_png = os.path.join(results_dir, 'results.png')

if os.path.exists(results_png):
    print('📈 Training curves:')
    display(IPImage(filename=results_png, width=900))
else:
    print(f'results.png not found at {results_png}')

# Also show confusion matrix
conf_matrix = os.path.join(results_dir, 'confusion_matrix.png')
if os.path.exists(conf_matrix):
    print('\n🔢 Confusion Matrix:')
    display(IPImage(filename=conf_matrix, width=600))

---
## 6. Run Inference on Test Images
See your model actually detecting parking spaces.

In [ ]:
import numpy as np

# Grab test images
test_imgs = glob.glob(os.path.join(DATASET_PATH, 'test/images/*.jpg'))
test_imgs += glob.glob(os.path.join(DATASET_PATH, 'test/images/*.png'))

if not test_imgs:
    # Fall back to valid images
    test_imgs = glob.glob(os.path.join(DATASET_PATH, 'valid/images/*.jpg'))

sample_imgs = random.sample(test_imgs, min(6, len(test_imgs)))

CLASS_NAMES  = {0: 'occupied', 1: 'empty'}
CLASS_COLORS = {0: (220, 50,  50),   # red  = occupied
                1: (50,  200, 50)}    # green = empty

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, img_path in zip(axes.flatten(), sample_imgs):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Run inference
    preds = best_model(img_path, verbose=False)[0]

    occupied = 0
    empty    = 0

    for box in preds.boxes:
        cls   = int(box.cls[0])
        conf  = float(box.conf[0])
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        color = CLASS_COLORS.get(cls, (255,255,0))
        label = f"{CLASS_NAMES.get(cls,'?')} {conf:.2f}"

        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), color, 2)
        cv2.putText(img_rgb, label, (x1, y1-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

        if cls == 0: occupied += 1
        else:        empty    += 1

    total = occupied + empty
    title = f'Occupied: {occupied} | Empty: {empty} | Total: {total}'
    ax.imshow(img_rgb)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')

plt.suptitle('ParkVision – Model Predictions (Red=Occupied, Green=Empty)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import time

# Measure inference speed (target: < 1 second per image)
test_img = sample_imgs[0]
N_RUNS   = 20

# Warmup
_ = best_model(test_img, verbose=False)

# Timed runs
start = time.time()
for _ in range(N_RUNS):
    best_model(test_img, verbose=False)
elapsed = time.time() - start

avg_ms = (elapsed / N_RUNS) * 1000
avg_s  = elapsed / N_RUNS

print('=== INFERENCE SPEED ===')
print(f'Average per image: {avg_ms:.1f} ms  ({avg_s:.3f} s)')
print(f'Target:            < 1000 ms (1 second)')

if avg_s < 1.0:
    print(f'✅ Meets speed target!')
else:
    print(f'⚠️  Too slow — switch to yolov8n for faster inference')

---
## 7. Save Model to Google Drive
**Critical step — always run this before closing Colab!**

In [ ]:
import shutil

DRIVE_SAVE_DIR = '/content/drive/MyDrive/ParkVision/'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Save best weights
src = BEST_MODEL_PATH
dst = os.path.join(DRIVE_SAVE_DIR, 'best.pt')
shutil.copy(src, dst)
print(f'✅ Best model weights saved → {dst}')

# Save full training run folder (includes curves, confusion matrix, etc.)
run_dir     = f"ParkVision/{CONFIG['name']}"
run_dst     = os.path.join(DRIVE_SAVE_DIR, CONFIG['name'])
if os.path.exists(run_dst):
    shutil.rmtree(run_dst)
shutil.copytree(run_dir, run_dst)
print(f'✅ Full training run saved → {run_dst}')

print('\n📁 Contents saved to Drive:')
for f in os.listdir(DRIVE_SAVE_DIR):
    print(f'   {f}')

---
## 🛠️ Troubleshooting Reference

| Problem | Fix |
|---------|-----|
| `CUDA out of memory` | Change `batch` from 16 → 8 in CONFIG |
| mAP50 stuck below 0.70 | Switch to `yolov8m.pt` and retrain |
| Training very slow | Make sure GPU is selected (Runtime → Change Runtime Type) |
| Colab disconnects | Save checkpoints to Drive — they survive disconnects |
| Dataset not found | Re-run cell 2 to re-download from Roboflow |
| Loss not decreasing | Try lowering `lr0` to 0.0005 |
| Loss jumping around | Try increasing `batch` to 32 if you have VRAM |

---
## ✅ What To Do Next

Once your mAP50 is ≥ 0.85:
1. Screenshot your results table — you'll need it for your presentation
2. Save this notebook to GitHub (`notebooks/02_train_model.ipynb`)
3. Move to `03_demo.ipynb` to build the visual overlay